<a href="https://colab.research.google.com/github/ValentinaEmili/Texture-synthesis/blob/main/VQ_VAE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install lpips

In [3]:
import os
from PIL import Image
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.nn.functional as F
import torch
import torch.optim as optim
from torchvision.models import vgg16
import lpips
import matplotlib.pyplot as plt
import time
import numpy as np

In [4]:
train_transform = transforms.Compose([
        transforms.Resize(512),
        transforms.RandomCrop(512),
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
        ])

eval_transform = transforms.Compose([
        transforms.Resize(512),
        transforms.CenterCrop(512),
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
        ])


class DTD_Dataset(Dataset):
    def __init__(self, root, file_list, transform=None, class_to_idx=None):
        self.root = root
        self.transform = transform

        with open(file_list, mode='r', encoding='utf-8') as f:
            self.files = [line.strip() for line in f if line.strip()]

        if class_to_idx is None:
          unique_classes = sorted({os.path.normpath(p).split(os.sep)[0] for p in self.files})
          self.class_to_idx = {class_name: i for i, class_name in enumerate(unique_classes)}
        else:
          self.class_to_idx = class_to_idx

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        relative_path = self.files[idx]
        image_path = os.path.join(self.root, relative_path)
        img = Image.open(image_path).convert('RGB')
        rel_norm = os.path.normpath(relative_path)
        string_label = rel_norm.split(os.sep, 1)[0]
        label = self.class_to_idx[string_label]

        if self.transform:
            img = self.transform(img)

        return img, label

In [5]:
path_images = "drive/MyDrive/DeepLearning/dtd/images"
path_labels = "drive/MyDrive/DeepLearning/dtd/labels"
train_dataset = DTD_Dataset(path_images, os.path.join(path_labels, "train1.txt"), train_transform)
class_to_idx = train_dataset.class_to_idx
val_dataset = DTD_Dataset(path_images, os.path.join(path_labels, "val1.txt"), eval_transform, class_to_idx=class_to_idx)
test_dataset = DTD_Dataset(path_images, os.path.join(path_labels, "test1.txt"), eval_transform, class_to_idx=class_to_idx)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers=2)

In [6]:
class Residual(nn.Module):
    def __init__(self, num_channels):
        super().__init__()

        self.block = nn.Sequential(

            nn.Conv2d(num_channels, num_channels, kernel_size=3, stride=1, padding=1),
            nn.GroupNorm(32, num_channels),
            nn.ReLU(inplace=True),

            nn.Conv2d(num_channels, num_channels, kernel_size=3, stride=1, padding=1),
            nn.GroupNorm(32, num_channels)
        )

    def forward(self, x):
        return x + self.block(x)

class Encoder(nn.Module):
    def __init__(self, hidden_dim=128, embedding_dim=128, num_channels=3):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(3, hidden_dim // 2, kernel_size=4, stride=2, padding=1),                      # 512 -> 256
            nn.GroupNorm(32, hidden_dim // 2),
            nn.ReLU(),

            nn.Conv2d(hidden_dim // 2, hidden_dim, kernel_size=4, stride=2, padding=1),             # 256 -> 128
            nn.GroupNorm(32, hidden_dim),
            nn.ReLU(),

            nn.Conv2d(hidden_dim, hidden_dim * 2, kernel_size=4, stride=2, padding=1),              # 128 -> 64
            nn.GroupNorm(32, hidden_dim * 2),
            nn.ReLU(),

            nn.Conv2d(hidden_dim * 2, hidden_dim * 4, kernel_size=4, stride=2, padding=1),          # 64 -> 32
            nn.GroupNorm(32, hidden_dim * 4),
            nn.ReLU(),

            nn.Conv2d(hidden_dim * 4, embedding_dim, kernel_size=3, stride=1, padding=1),          # 32 -> 32
            nn.GroupNorm(32, embedding_dim)
        )
        self.residual = Residual(embedding_dim)

    def forward(self, x):
        x = self.conv(x)
        return self.residual(x)

class Decoder(nn.Module):
    def __init__(self, embedding_dim=128, hidden_dim=128, num_channels=3):
        super().__init__()

        self.residual = Residual(embedding_dim)

        self.conv = nn.Sequential(
            nn.ConvTranspose2d(embedding_dim, hidden_dim * 4, kernel_size=3, stride=1, padding=1),             # 32 -> 32
            nn.GroupNorm(32, hidden_dim * 4),
            nn.ReLU(),

            nn.ConvTranspose2d(hidden_dim * 4, hidden_dim * 2, kernel_size=4, stride=2, padding=1),             # 32 -> 64
            nn.GroupNorm(32, hidden_dim * 2),
            nn.ReLU(),

            nn.ConvTranspose2d(hidden_dim * 2, hidden_dim, kernel_size=4, stride=2, padding=1),              # 64 -> 128
            nn.GroupNorm(32, hidden_dim),
            nn.ReLU(),

            nn.ConvTranspose2d(hidden_dim, hidden_dim // 2, kernel_size=4, stride=2, padding=1),          # 128 -> 256
            nn.GroupNorm(32, hidden_dim // 2),
            nn.ReLU(),

            nn.ConvTranspose2d(hidden_dim // 2, num_channels, kernel_size=4, stride=2, padding=1)           # 256 -> 512
        )
    def forward(self, x):
        x = self.residual(x)
        x = self.conv(x)
        return torch.tanh(x)

class VectorQuantizer(nn.Module):
    def __init__(self, num_embeddings=512, embedding_dim=128, commitment_cost=0.25):
        super().__init__()
        self.embedding_dim = embedding_dim
        self.num_embeddings = num_embeddings
        self.commitment_cost = commitment_cost

        self.embeddings = nn.Embedding(num_embeddings, embedding_dim)
        #self.embeddings.weight.data.uniform_(-1.0/self.num_embeddings, 1.0/self.num_embeddings)
        #self.embeddings.weight.data.uniform_(-1.0/self.embedding_dim, 1.0/self.embedding_dim)
        self.embeddings.weight.data.uniform_(-0.1, 0.1)

    def forward(self, z):
        z_flattened = z.permute(0, 2, 3, 1).contiguous()
        z_flattened = z_flattened.view(-1, self.embedding_dim)

        distances = (torch.sum(z_flattened**2, dim=1, keepdim=True)
                     + torch.sum(self.embeddings.weight**2, dim=1)
                     - 2 * torch.matmul(z_flattened, self.embeddings.weight.t()))

        encoding_indices = torch.argmin(distances, dim=1)
        encodings = F.one_hot(encoding_indices, self.num_embeddings).float()

        quantized = torch.matmul(encodings, self.embeddings.weight)
        quantized = quantized.view(z.shape[0], z.shape[2], z.shape[3], self.embedding_dim)
        quantized = quantized.permute(0, 3, 1, 2).contiguous()

        e_latent_loss = F.mse_loss(quantized.detach(), z)
        q_latent_loss = F.mse_loss(quantized, z.detach())
        loss = q_latent_loss + self.commitment_cost * e_latent_loss

        quantized = z + (quantized - z).detach()

        avg_probs = torch.mean(encodings, dim=0)
        perplexity = torch.exp(-torch.sum(avg_probs * torch.log(avg_probs + 1e-10)))

        active_codes = torch.unique(encoding_indices).numel()

        return quantized, loss, perplexity, active_codes

class VQ_VAE(nn.Module):
    def __init__(self, num_embeddings=512, embedding_dim=128, hidden_dim=128, commitment_cost=0.25):
        super().__init__()
        self.encoder = Encoder(hidden_dim, embedding_dim)
        self.vq = VectorQuantizer(num_embeddings, embedding_dim, commitment_cost)
        self.decoder = Decoder(embedding_dim, hidden_dim)

    def forward(self, x):
        z = self.encoder(x)
        quantized, vq_loss, perplexity, active_codes = self.vq(z)
        x_recon = self.decoder(quantized)
        return x_recon, vq_loss, perplexity, active_codes


perceptual loss compares features maps, instead of raw pixels:

In [12]:
class PerceptualLoss(nn.Module):
    def __init__(self):
        super().__init__()
        VGG16 = vgg16(weights='DEFAULT').features
        self.slice = nn.Sequential(*list(VGG16.children())[:16]).eval()

        for param in self.slice.parameters():
            param.requires_grad = False

        # ImageNet normalization statistics
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def normalize(self, x):
      x = (x + 1.0) / 2.0
      return (x - self.mean) / self.std

    def forward(self, real, fake):
        real = self.slice(self.normalize(real))
        fake = self.slice(self.normalize(fake))
        return F.mse_loss(fake, real)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = VQ_VAE().to(device)
#optimizer = optim.Adam(model.parameters(), lr=1e-4, betas=(0.5, 0.9))  # for VQ-GAN
#perceptual_loss_fn = PerceptualLoss().to(device)
optimizer = optim.Adam(model.parameters(), lr=5e-5, betas=(0.9, 0.999)) # for VQ-VAE
perceptual_loss_fn = lpips.LPIPS(net='vgg').to(device)

In [9]:
def visual_validation(model, val_loader, device):
  model.eval()
  encoding_indices = []
  unique_indices = []
  visual_samples = None

  num_embeddings = model.vq.num_embeddings
  embedding_dim = model.vq.embedding_dim
  embeddings = model.vq.embeddings.weight

  with torch.no_grad():
    for batch_idx, (data, _) in enumerate(val_loader):
      data = data.to(device)
      recon_batch, vq_loss = model(data)

      visual_samples = (data.cpu(), recon_batch.cpu())

      z = model.encoder(data)
      z_flattened = z.permute(0, 2, 3, 1).contiguous().view(-1, embedding_dim)

      distances = (torch.sum(z_flattened**2, dim=1, keepdim=True)
                    + torch.sum(embeddings**2, dim=1)
                    - 2 * torch.matmul(z_flattened, embeddings.t()))

      encoding_indices = torch.argmin(distances, dim=1)

  # percentage of used vectors
  unique_indices = torch.unique(encoding_indices)
  util_percen = len(unique_indices) / num_embeddings * 100

  # perplexity
  counts = torch.bincount(encoding_indices, minlength=num_embeddings).float()
  probs = counts / counts.sum()
  perplexity = torch.exp(-torch.sum(probs * torch.log(probs + 1e-10)))

  print(f"Total Codebook Size:    {num_embeddings}")
  print(f"Unique Vectors Used:    {len(unique_indices)} / {num_embeddings}")
  print(f"Codebook Utilization:   {util_percen:.2f}%")
  print(f"Codebook Perplexity:    {perplexity.item():.2f} (Higher is better)")

  # visual reconstruction plotting
  real_imgs, recon_imgs = visual_samples
  num_displayed_imgs = min(4, real_imgs.shape[0])

  fig, axes = plt.subplots(2, num_displayed_imgs, figsize=(num_displayed_imgs * 3, 6))

  for i in range(num_displayed_imgs):
    real_img = real_imgs[i].permute(1, 2, 0).numpy()
    recon_img = recon_imgs[i].permute(1, 2, 0).numpy()

    real_plot = ((real_img + 1) / 2).clip(0, 1)
    recon_plot = ((recon_img + 1) / 2).clip(0, 1)

    axes[0, i].imshow(real_plot)
    axes[0, i].set_title(f"original {i+1}")
    axes[0, i].axis('off')

    axes[1, i].imshow(recon_plot)
    axes[1, i].set_title(f"reconstructed {i+1}")
    axes[1, i].axis('off')

  plt.show()

In [24]:
model.train()
save_path = '/content/drive/MyDrive/DeepLearning/dtd/checkpoints/vq_vae'
os.makedirs(save_path, exist_ok=True)
epochs = 40

for epoch in range(epochs):
    epoch_start = time.time()
    total_loss, epoch_perplexity, avg_active_codes = 0.0, 0.0, 0.0
    total_percept, total_recon, total_vq = 0.0, 0.0, 0.0
    for batch_idx, (data, _) in enumerate(train_loader):
        data = data.to(device)
        optimizer.zero_grad()
        data_recon, vq_loss, perplexity, active_codes = model(data)                         # codebook loss

        data = F.interpolate(data, size=(256, 256), mode='bilinear', align_corners=False)
        data_recon = F.interpolate(data_recon, size=(256, 256), mode='bilinear', align_corners=False)

        recon_loss = F.mse_loss(data_recon, data)                                           # reconstruction loss
        percept_loss = perceptual_loss_fn(data_recon, data).mean()                          # perceptual loss
        loss = percept_loss + vq_loss + recon_loss * 0.2
        loss.backward()

        optimizer.step()

        total_loss += loss.item()
        epoch_perplexity += perplexity.item()
        avg_active_codes += active_codes

        total_percept += percept_loss
        total_recon += recon_loss
        total_vq += vq_loss
    #visual_validation(model, val_loader, device)
    print(f'====> Epoch: {epoch} | Average loss: {total_loss / len(train_loader):.4f} | Perplexity: {epoch_perplexity/len(train_loader):.2f} | Active codes: {avg_active_codes/len(train_loader):.2f}')
    print(f'----- Perceptual Loss: {total_percept / len(train_loader):.4f} | Reconstruction Loss: {total_recon / len(train_loader):.4f} | Codebook Loss: {total_vq / len(train_loader):.4f} \n')
    # save model
    #epoch_save_path = os.path.join(save_path, f"epoch_{epoch}.pth")
    #torch.save(model.state_dict(), epoch_save_path)


====> Epoch: 0 | Average loss: 2.4100 | Perplexity: 26.69 | Active codes: 60.92
----- Perceptual Loss: 0.5186 | Reconstruction Loss: 0.1006 | Codebook Loss: 1.8713 

====> Epoch: 1 | Average loss: 2.6586 | Perplexity: 26.73 | Active codes: 59.13
----- Perceptual Loss: 0.5132 | Reconstruction Loss: 0.0992 | Codebook Loss: 2.1255 

====> Epoch: 2 | Average loss: 2.7678 | Perplexity: 27.18 | Active codes: 57.50
----- Perceptual Loss: 0.5059 | Reconstruction Loss: 0.0970 | Codebook Loss: 2.2425 

====> Epoch: 3 | Average loss: 2.8894 | Perplexity: 27.23 | Active codes: 56.21
----- Perceptual Loss: 0.5008 | Reconstruction Loss: 0.0971 | Codebook Loss: 2.3691 

====> Epoch: 4 | Average loss: 3.0048 | Perplexity: 27.74 | Active codes: 54.74
----- Perceptual Loss: 0.4936 | Reconstruction Loss: 0.0924 | Codebook Loss: 2.4927 

====> Epoch: 5 | Average loss: 3.0574 | Perplexity: 28.06 | Active codes: 53.70
----- Perceptual Loss: 0.4886 | Reconstruction Loss: 0.0919 | Codebook Loss: 2.5504 

====

In [ ]:
epochs = 40
save_path = '/content/drive/MyDrive/DeepLearning/dtd/checkpoints/vq_vae'

for epoch in range(epochs):
  epoch_save_path = os.path.join(save_path, f"epoch_{epoch}.pth")
  model = VQ_VAE().to(device)
  model.load_state_dict(torch.load(epoch_save_path, weights_only=True))
  visual_validation(model, val_loader, device)
  break
